# Phase 8 — Adaptive Learning Under Distribution Shift

## 1. Objective
In Phase 7, we discovered that while our self-supervised Temporal Transformer maintains high latent separability (ROC-AUC 0.9575) under distribution shift, static decision thresholding on unadapted models leads to an explosion of False Positives ($FPR = 91.33\%$) and severe F1 degradation ($F1 = 0.3383$).

The objective of Phase 8 is to determine experimentally:
> **"Can controlled adaptation recover anomaly-detection performance after a distribution shift without catastrophic forgetting of the original operating distribution?"**

### System Architecture
```
Incoming Sensor Data
        ↓
Preprocessing (StandardScaler)
        ↓
Distribution Shift Detector (KS Test, Threshold = 0.009148)
        ↓
   ┌────────────────────────────────┐
   │                                │
Stable                       Shift Detected (Score = 0.4218)
   │                                │
   ↓                                ↓
Inference (Original Model)   Controlled Adaptation (Replay Buffer r=0.20)
                                    ↓
                             Safety Checks & Acceptance Criteria
                                    ↓
                             Promoted Adapted Model
                                    ↓
                             Inference & Robust Anomaly Detection
```


## 2. Existing Model and Shift Detector
We inspect the Phase 6 SSL-pretrained Temporal Transformer and the Phase 7 Kolmogorov-Smirnov distribution shift detector.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '../src')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from config import SENSOR_COLS, RESULTS_DIR, FIGURES_DIR, DATA_PROC, MODELS_DIR

ADAPT_RES = RESULTS_DIR / 'adaptation'
ADAPT_FIGS = FIGURES_DIR / 'adaptation'
ADAPT_MOD = MODELS_DIR / 'adaptation'

with open(RESULTS_DIR / 'drift' / 'detector_config.json') as f:
    det_cfg = json.load(f)

print(f"Shift Detector:        {det_cfg['method']}")
print(f"Detection Threshold:   {det_cfg['threshold']:.6f}")
print(f"Features Monitored:    {len(det_cfg['feature_cols'])}")

with open(MODELS_DIR / 'ssl' / 'ssl_config.json') as f:
    ssl_cfg = json.load(f)
print(f"Primary Model:         SSL Transformer (d_model={ssl_cfg['d_model']}, layers={ssl_cfg['num_layers']})")


## 3. Adaptation-Data Preparation
We inspect the chronological data splitting protocol. Shifted data is split into:
1. **Adaptation Pool (Nov 1–30, 2018)**:
   - `adapt_train_df` (Nov 1–21, 6,048 rows): for incremental fine-tuning.
   - `adapt_val_df` (Nov 22–30, 2,592 rows): for validation and decision threshold recalibration.
2. **Untouched Evaluation Subset (Dec 1–31, 2018, 8,928 rows)**:
   - Evaluated strictly *after* adaptation to measure out-of-sample generalization.


In [ ]:
with open(ADAPT_RES / 'adaptation_summary.json') as f:
    adapt_summary = json.load(f)

print(f"Shift Trigger Status:   Shift Score = {adapt_summary['shift_score']:.6f} > {adapt_summary['shift_threshold']:.6f}")
print(f"Adaptation Permitted:   {adapt_summary['shift_detected']}")
print(f"Adaptation Strategy:    {adapt_summary['adaptation_strategy']}")
print(f"Replay Buffer Ratio:    {adapt_summary['replay_ratio']*100:.0f}%")
print(f"Adaptation Duration:    {adapt_summary['adaptation_time_s']:.2f} seconds")


## 4. Data-Leakage Prevention
To guarantee rigorous methodology and prevent data leakage:
- **Strict Temporal Partitioning**: Adaptation is trained exclusively on early shifted timestamps (November). December data is held out entirely as unseen evaluation data.
- **No Test Labels in Training**: The candidate model never sees any sequence from the December evaluation period.
- **No Evaluation on Training Data**: We never train and test on the same shifted sequences.


In [ ]:
print("Data Partitioning Schema:")
print("  Phase 2 Reference Train: Jan 1 – Aug 31, 2018 (69,984 rows)")
print("  Phase 2 Reference Val:   Sep 1 – Oct 31, 2018 (17,568 rows)")
print("  Phase 8 Adaptation Pool: Nov 1 – Nov 30, 2018 (8,640 rows -> 6,048 train / 2,592 val)")
print("  Phase 8 Final Eval Pool: Dec 1 – Dec 31, 2018 (8,928 rows held-out)")


## 5. Adaptation Strategy: Controlled Incremental Fine-Tuning with Replay Buffer
- **Mechanism**: Fine-tune the full SSL Transformer parameters for a small number of epochs ($E=5$) using AdamW at a low learning rate ($\eta = 10^{-4}$).
- **Replay Buffer**: To counteract catastrophic forgetting of the nominal operating regime, every adaptation batch mixes $80\%$ incoming shifted sequences with $20\%$ randomly sampled sequences from the original training distribution.
- **Decision Threshold Recalibration**: Rather than keeping a static threshold from the unshifted regime, the threshold is recalibrated on `adapt_val_df` to maximize F1 ($	au_{\text{adapted}} = 0.0100$).


## 6. Adaptation Configuration
Hyperparameters used during controlled adaptation:
- Optimizer: `AdamW`
- Learning rate: $1 \times 10^{-4}$
- Weight decay: $1 \times 10^{-4}$
- Batch size: 128
- Adaptation epochs: 5
- Replay ratio: 0.20


In [ ]:
print("Adaptation Training Config:")
print(f"  Epochs:         {adapt_summary['epochs']}")
print(f"  Learning Rate:  {adapt_summary['learning_rate']}")
print(f"  Replay Ratio:   {adapt_summary['replay_ratio']}")
print(f"  Calibrated Threshold: {adapt_summary['candidate_threshold']}")


## 7. Model Fine-Tuning Curves
We inspect the training loss progression and validation F1 score across the 5 adaptation epochs.


In [ ]:
# Display adaptation training curves
Image.open(ADAPT_FIGS / 'adaptation_training_curves.png')


## 8. Before vs. After Adaptation Evaluation (4-Way Comparison)
We compare the performance of the Original SSL Transformer vs. the Adapted SSL Transformer across both Normal and Shifted evaluation conditions on the held-out December test period.


In [ ]:
comp_df = pd.read_csv(ADAPT_RES / 'final_comparison_table.csv')
display(comp_df)

# Display 4-way comparison bar chart
Image.open(ADAPT_FIGS / 'four_way_comparison.png')


## 9. Original-Distribution Evaluation (Catastrophic-Forgetting Analysis)
Did fine-tuning on shifted data cause catastrophic forgetting of the nominal normal distribution?

As measured on the held-out December normal evaluation set:
- **Original Model on Normal**: F1 = **0.9570**, Precision = 0.9289, Recall = 0.9869, ROC-AUC = 0.9961, FPR = 1.76%
- **Adapted Model on Normal**:  F1 = **0.9764**, Precision = 0.9695, Recall = 0.9833, ROC-AUC = 0.9962, FPR = 0.72%

**Finding**: No catastrophic forgetting occurred. In fact, normal condition F1 improved by **+0.0194**, and false alarm rate decreased from $1.76\%$ to $0.72\%$ thanks to the regularizing effect of the $20\%$ replay buffer.


In [ ]:
# Display Before vs After F1 comparison
Image.open(ADAPT_FIGS / 'f1_before_after.png')


## 10. Shifted-Distribution Evaluation (Recovery)
On the held-out December shifted evaluation set:
- **Original Model on Shifted**: F1 = **0.3383**, Precision = 0.2036, Recall = 1.0000, FPR = 91.33%
- **Adapted Model on Shifted**:  F1 = **0.8623**, Precision = 0.7678, Recall = 0.9833, FPR = 6.94%

**Key Recovery Metrics**:
- **F1 Recovery**: **$+0.5240$** (from 0.3383 to 0.8623)
- **Precision Recovery**: **$+0.5642$** (from 20.36% to 76.78%)
- **FPR Suppression**: **$-84.39\%$** absolute reduction (from 91.33% to 6.94%—false alarms reduced by over 92%!)
- **ROC-AUC**: **0.9930** (up from 0.9592)
- **PR-AUC**: **0.9876** (up from 0.8795)


In [ ]:
# Display Precision & Recall trajectory
Image.open(ADAPT_FIGS / 'precision_recall_recovery.png')


## 11. False Alarm Suppression Analysis
The primary failure mode under distribution shift was false alarm surge due to elevated baselines. We visualize the drastic reduction in False Positive Rate achieved by adaptation.


In [ ]:
# Display FPR reduction bar chart
Image.open(ADAPT_FIGS / 'fpr_reduction.png')


## 12. Adaptation Cost Profiling
We measure the computational and operational costs associated with adaptation.


In [ ]:
cost_df = pd.read_csv(ADAPT_RES / 'adaptation_cost.csv')
display(cost_df)


- **Adaptation Duration**: **68.48 seconds** on CPU for 5 epochs over 7,501 sequence windows.
- **Inference Latency**: **2.06 ms/sequence** (essentially identical to before adaptation: 2.14 ms).
- **Model Checkpoint Size**: **95.70 KB** (19,313 parameters, zero architecture bloat).


## 13. Error Analysis
We examine false positive and false negative transitions on the full shifted test period.


In [ ]:
err_df = pd.read_csv(ADAPT_RES / 'error_analysis.csv')
display(err_df)


### Mechanistic Observations:
1. **False Alarm Extinction**: On the full shifted dataset, false positive alarms plummeted from **12,863 to 935** (an absolute elimination of **11,928 false alarms**). Adaptation realigns the model's self-attention representations to the new baseline offset.
2. **Minimal False Negative Trade-off**: The model introduced only 70 missed sequences across the entire shifted dataset while correctly detecting over 3,480 anomalies (Recall = 98.33%).


## 14. Results Summary
The adaptation pipeline succeeded across all predefined criteria:
- Shift detector successfully triggered adaptation when shift score reached $0.4218$ ($	au = 0.0091$).
- Candidate model satisfied both safety criteria:
  - $\Delta F1_{\text{shifted\_val}} = +0.3340 > 0$
  - $\Delta F1_{\text{ref\_val}} = +0.0017 \ge -0.05$
- Candidate was promoted to `models/adaptation/ssl_transformer_adapted.pt`.
- Held-out December test evaluation proved substantial generalization and robustness recovery.


## 15. Limitations
1. **Periodic vs. Streaming**: The current protocol uses periodic batch adaptation (evaluating November and testing December), rather than true streaming step-by-step updates.
2. **Replay Buffer Storage**: Replay requires retaining a fraction ($20\%$) of historical reference data in memory.
3. **Supervised Adaptation Labels**: The adaptation subset leveraged ground-truth anomaly labels. In pure unsupervised deployments, pseudo-labeling or self-supervised masked reconstruction pre-adaptation would be required.


## 16. Findings & Handoff to Phase 9

### Key Takeaways:
1. **Adaptation is Highly Effective**: Controlled fine-tuning recovered F1 from 0.3383 to 0.8623 on unseen shifted data, resolving the false alarm crisis caused by distribution shift.
2. **Replay Prevents Catastrophic Forgetting**: A modest $20\%$ replay buffer completely protected the nominal normal distribution performance ($F1 = 0.9764$ vs. $0.9570$).
3. **Safety Verification is Essential**: Systematic candidate model gating ensures corrupted or overfitted models are never deployed without verified validation.
4. **Feasible Computational Footprint**: With an adaptation training time under 70 seconds on standard CPU, periodic adaptation is computationally practical for water utility SCADA systems.
